# Medical FAQ Semantic Search

## 1. Project objective
This notebook loads a raw CSV of medical FAQs, cleans it using functions, splits the answers into chunks, and processes the text into a **semantic search** with Chroma. NumPy is used to measure the data and to flag weak matches.

It answers three questions: **how many FAQs remain after cleaning**, **how many chunks do we store**, and **does a question written in everyday words find the right FAQ** (even when it shares almost no words with it).

**Steps:** load -> inspect -> clean -> chunk -> embed and store -> search -> results -> export -> tests -> conclusion

> This is a learning project from the AASTU AI/ML cohort (Week 2 lab). It is not medical advice.

**Google Colab:** click **Runtime -> Run all** and upload `faqs.csv` when asked.

## 2. Install, imports and settings
Colab does not include Chroma, so we install it first. A pip warning about `google-adk` or `opentelemetry` can be ignored.

In [ ]:
!pip install -q chromadb

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import chromadb

# Settings (practice values, change them and rerun)
CSV_PATH = "faqs.csv"
REQUIRED_COLUMNS = ("question", "answer")
CHUNK_SIZE = 40          # words per chunk
OVERLAP = 10             # words shared between neighbouring chunks
TOP_K = 3                # matches returned per query
WEAK_MATCH_DISTANCE = 1.0  # a distance at or above this is flagged as a weak match
OUTPUT_DIR = Path("outputs")

QUERIES = [
    "I twisted my foot playing football and it is swollen",
    "I keep vomiting and have stomach cramps after eating",
    "someone collapsed and is not breathing",
    "how do I avoid mosquito bites",
]

print("Libraries imported! NumPy", np.__version__, "| pandas", pd.__version__)

## 3. Load the dataset
The CSV file must be in the same folder as this notebook. If you run this in Google Colab, the cell below will ask you to upload it.

In [ ]:
def load_faqs(path):
    """Read the CSV file and return a table (DataFrame). Return None if the file is missing."""
    try:
        data = pd.read_csv(path)
    except FileNotFoundError:
        print("Error: file not found ->", path)
        return None

    missing_columns = [c for c in REQUIRED_COLUMNS if c not in data.columns]
    if missing_columns:
        raise ValueError(f"CSV must contain these columns: {', '.join(REQUIRED_COLUMNS)}")
    return data


raw = load_faqs(CSV_PATH)

if raw is None:
    try:
        from google.colab import files
        print("Please upload faqs.csv")
        files.upload()
        raw = load_faqs(CSV_PATH)
    except ImportError:
        print("Put the CSV in the same folder as this notebook.")

## 4. Inspect the raw data
Before cleaning anything, look at the data and find the problems yourself.

In [ ]:
print("Size (rows, columns):", raw.shape)
print()
print(raw.head(5))

In [ ]:
print("Column types:")
print(raw.dtypes)
print()
print("Missing values per column:")
print(raw.isna().sum())
print()
print("Duplicate rows:", raw.duplicated().sum())
print("Repeated questions:", raw["question"].duplicated().sum())

**What I notice:**
- One answer is missing.
- One question appears twice (same question, same answer).
- Two questions have extra spaces at the start, at the end or between words.

In [ ]:
missing = raw[raw["answer"].isna()]
repeated = raw[raw["question"].duplicated(keep=False)]
spacey = raw[raw["question"].str.contains(r"^\s|\s$|\s{2,}", regex=True)]

print("Rows with a missing answer:")
print(missing["question"].to_string())
print()
print("Rows with a repeated question:")
print(repeated["question"].to_string())
print()
print("Questions with extra spaces:")
print(spacey["question"].apply(repr).to_string())

## 5. Cleaning functions
Each function does only one job. `clean_data` runs them in order and also counts the rows it rejects. The order matters: spaces are fixed **before** removing duplicates, so two questions that only differ by spaces count as the same question.

In [ ]:
def clean_text_columns(data):
    """Remove extra spaces at the ends and between words in the question and answer."""
    data = data.copy()
    for col in REQUIRED_COLUMNS:
        data[col] = (
            data[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    return data


def remove_missing(data):
    """Drop rows where the question or the answer is missing or empty."""
    data = data.dropna(subset=list(REQUIRED_COLUMNS))
    data = data[(data["question"] != "") & (data["answer"] != "")]
    return data


def remove_duplicate_questions(data):
    """Keep each question once (the first one)."""
    return data.drop_duplicates(subset=["question"])


def add_faq_id(data):
    """Give every remaining FAQ a clean number: 0, 1, 2, ..."""
    data = data.reset_index(drop=True)
    data["faq_id"] = data.index
    return data


def clean_data(data):
    """Run all cleaning steps in order. Return the clean table and a count of rejected rows."""
    rejected = Counter()

    data = clean_text_columns(data)

    before = len(data)
    data = remove_missing(data)
    rejected["missing or empty question/answer"] = before - len(data)

    before = len(data)
    data = remove_duplicate_questions(data)
    rejected["duplicate question"] = before - len(data)

    data = add_faq_id(data)
    return data, rejected

## 6. Run the cleaning and compare raw vs clean

In [ ]:
clean, rejected = clean_data(raw)

print("Raw size:  ", raw.shape)
print("Clean size:", clean.shape)
print()
print("Rejected rows:")
for reason, count in rejected.items():
    print(f"  {reason}: {count}")

assert len(raw) == len(clean) + sum(rejected.values())

In [ ]:
print("Raw question 8:  ", repr(raw.loc[8, "question"]))
print("Clean has it as: ", repr(clean[clean["question"].str.startswith("How long should I wash")]["question"].iloc[0]))
print()
print("Clean column types:")
print(clean.dtypes)

## 7. Chunking with NumPy
Long answers are cut into overlapping pieces of words (the overlap stops a sentence from being cut in half). The question is added to every chunk, so a piece still knows what it is answering.

First, NumPy measures how long the answers are. This tells us if our chunk size makes sense: the lecture suggests about 200 words, but if every answer is shorter than that, nothing would ever be split.

In [ ]:
def answer_word_counts(data):
    """Number of words in each answer, as a NumPy array."""
    return data["answer"].str.split().str.len().to_numpy()


def describe_lengths(counts, chunk_size):
    """Use NumPy to summarise the answer lengths."""
    return {
        "answers": len(counts),
        "shortest": int(np.min(counts)),
        "average": float(np.mean(counts)),
        "median": float(np.median(counts)),
        "longest": int(np.max(counts)),
        "longer_than_chunk": int(np.sum(counts > chunk_size)),
    }


counts = answer_word_counts(clean)
length_stats = describe_lengths(counts, CHUNK_SIZE)

for name, value in length_stats.items():
    print(f"  {name}: {value:.1f}" if isinstance(value, float) else f"  {name}: {value}")
print()
print(f"With CHUNK_SIZE={CHUNK_SIZE}, {length_stats['longer_than_chunk']} answers are split into more than one chunk.")
print(f"With CHUNK_SIZE=200, {int(np.sum(counts > 200))} answers would be split.")

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Split text into overlapping chunks of words."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text]

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])
        chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    return chunks


def build_chunks(data, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Turn each FAQ into one or more chunks, keeping the question as context."""
    ids, documents, metadatas = [], [], []
    for _, row in data.iterrows():
        pieces = chunk_text(row["answer"], chunk_size, overlap)
        for i, piece in enumerate(pieces):
            ids.append(f"faq{row['faq_id']}_chunk{i}")
            documents.append(f"Q: {row['question']} A: {piece}")
            metadatas.append({
                "faq_id": int(row["faq_id"]),
                "question": str(row["question"]),
                "chunk_index": i,
            })
    return ids, documents, metadatas


ids, documents, metadatas = build_chunks(clean)
print("FAQs:", len(clean), "| Chunks:", len(ids))
print()
print("Example chunk:")
print(documents[0])

## 8. Embed and store in Chroma
Chroma downloads a small embedding model (all-MiniLM-L6-v2) the first time. It turns each chunk into a vector of 384 numbers that describes its meaning. Texts with similar meaning get vectors that sit close together. The old collection is deleted first so a rerun starts clean.

In [ ]:
def build_index(ids, documents, metadatas, name="faqs"):
    """Embed the chunks and store them in a fresh Chroma collection."""
    client = chromadb.Client()  # in memory, fine for Colab

    try:
        client.delete_collection(name)
    except Exception:
        pass

    collection = client.get_or_create_collection(name=name)
    collection.add(ids=ids, documents=documents, metadatas=metadatas)
    return collection


collection = build_index(ids, documents, metadatas)
print("Chunks stored in Chroma:", collection.count())

## 9. Search functions
**Distance: lower means closer.** Chroma always returns exactly `TOP_K` results, even when the last ones are not relevant. So we use a NumPy Boolean mask to flag weak matches, the same idea as the alert threshold in the IoT lab (`WEAK_MATCH_DISTANCE` is a practice rule, not a scientific value).

In [ ]:
def search(collection, query, top_k=TOP_K):
    """Return the top_k closest chunks for a query."""
    results = collection.query(query_texts=[query], n_results=top_k)
    matches = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        matches.append({
            "document": doc,
            "question": meta["question"],
            "chunk_index": meta["chunk_index"],
            "distance": dist,
        })
    return matches


def flag_weak_matches(matches, threshold=WEAK_MATCH_DISTANCE):
    """Use NumPy to mark which matches are too far away. Returns a Boolean array."""
    distances = np.array([m["distance"] for m in matches], dtype=float)
    return distances >= threshold


def print_matches(query, matches):
    weak = flag_weak_matches(matches)
    print(f"\nQuery: {query}")
    for rank, (m, is_weak) in enumerate(zip(matches, weak), start=1):
        note = "  [weak match]" if is_weak else ""
        print(f"  {rank}. (distance {m['distance']:.3f}){note} {m['document']}")


def summarise_query(query, matches):
    """Use NumPy to summarise one query: best distance, weak matches, distinct FAQs."""
    distances = np.array([m["distance"] for m in matches], dtype=float)
    questions = np.array([m["question"] for m in matches])
    return {
        "query": query,
        "best_distance": round(float(np.min(distances)), 3),
        "weak_matches": int(np.sum(flag_weak_matches(matches))),
        "distinct_faqs": int(len(np.unique(questions))),
        "best_faq": matches[int(np.argmin(distances))]["question"],
    }

## 10. Results

### Top matches for each query

In [ ]:
all_matches = {}
for q in QUERIES:
    all_matches[q] = search(collection, q)
    print_matches(q, all_matches[q])

### Summary per query
`distinct_faqs` below 3 means two of the top matches are chunks of the **same** FAQ.

In [ ]:
summary = pd.DataFrame([summarise_query(q, m) for q, m in all_matches.items()])
summary

## 11. Export the results
The results are saved as CSV files in `outputs/`, so they can be reopened in Excel or Python.

In [ ]:
def matches_to_rows(all_matches):
    """Flatten the matches of every query into one list of rows."""
    rows = []
    for query, matches in all_matches.items():
        weak = flag_weak_matches(matches)
        for rank, (m, is_weak) in enumerate(zip(matches, weak), start=1):
            rows.append({
                "query": query,
                "rank": rank,
                "distance": round(m["distance"], 3),
                "weak_match": bool(is_weak),
                "faq_question": m["question"],
                "chunk_index": m["chunk_index"],
            })
    return rows


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUTPUT_DIR / "faqs_clean.csv", index=False)
pd.DataFrame(matches_to_rows(all_matches)).to_csv(OUTPUT_DIR / "search_results.csv", index=False)
summary.to_csv(OUTPUT_DIR / "query_summary.csv", index=False)

print("Wrote:", OUTPUT_DIR / "faqs_clean.csv")
print("Wrote:", OUTPUT_DIR / "search_results.csv")
print("Wrote:", OUTPUT_DIR / "query_summary.csv")
print("In Colab, you can download them from the Files panel on the left.")

## 12. Testing
Checking that the app does not crash on bad input and that each function does its one job.

In [ ]:
print("Test 1: wrong path")
print(load_faqs("this_file_does_not_exist.csv"))

In [ ]:
print("Test 2: a small table with every problem at once")

test_data = pd.DataFrame({
    "question": [
        "How do I treat a burn?",
        "How do I treat a burn?",       # duplicate
        "  How long   to wash hands? ",  # extra spaces
        "Is tap water safe?",           # missing answer
        "",                             # empty question
    ],
    "answer": [
        "Cool it under running water.",
        "Cool it under running water.",
        "At least 20 seconds.",
        None,
        "Some answer",
    ],
})

test_clean, test_rejected = clean_data(test_data)
print()
print(test_clean)
print(dict(test_rejected))

In [ ]:
print("Test 3: check the cleaning with asserts")

assert len(test_clean) == 2
assert "How long to wash hands?" in test_clean["question"].values
assert list(test_clean["faq_id"]) == [0, 1]
assert test_rejected["missing or empty question/answer"] == 2
assert test_rejected["duplicate question"] == 1
print("All cleaning tests passed")

In [ ]:
print("Test 4: chunking")

short_text = "one two three"
assert chunk_text(short_text) == [short_text]          # short text stays in one piece

long_words = [f"w{i}" for i in range(100)]
long_chunks = chunk_text(" ".join(long_words), chunk_size=40, overlap=10)

assert len(long_chunks) == 3
assert long_chunks[-1].split()[-1] == "w99"             # the last word is not lost
assert long_chunks[0].split()[-10:] == long_chunks[1].split()[:10]  # neighbours share 10 words
print("Number of chunks for 100 words:", len(long_chunks))
print("All chunking tests passed")

In [ ]:
print("Test 5: weak match flag")

fake_matches = [{"distance": 0.5}, {"distance": 1.0}, {"distance": 1.3}]
flags = flag_weak_matches(fake_matches)
print(flags)
assert list(flags) == [False, True, True]
print("Weak match test passed")

In [ ]:
print("Test 6: empty table")

empty = pd.DataFrame(columns=list(REQUIRED_COLUMNS))
empty_clean, empty_rejected = clean_data(empty)
if empty_clean.empty:
    print("The table is empty, nothing to process.")

In [ ]:
print("Test 7: check the real data")

assert len(raw) == 50, "Unexpected input: this lab expects the sample faqs.csv"
assert len(clean) == 48, "Expected 48 clean FAQs"
assert len(ids) == collection.count(), "Chroma should hold every chunk"
print("Raw =", len(raw), "| Clean =", len(clean), "| Chunks =", len(ids))
print("All sample checks passed")

## 13. Conclusion
- The raw file had **50 rows**. After cleaning, **48 rows** remained (1 row with a missing answer and 1 repeated question removed, extra spaces fixed in 2 questions).
- With `CHUNK_SIZE = 40` and `OVERLAP = 10`, the 48 FAQs became **79 chunks** in Chroma.
- The question "how do I avoid mosquito bites" found the malaria prevention FAQ as its best match, even though the two share almost no words. This is the point of semantic search: it matches meaning, not exact words.
- Chroma always returns `TOP_K` results, so the last ones can be weak. The NumPy flag marks them, and the summary table shows when two of the top matches are chunks of the same FAQ.

**My results (fill in after running):**
- Best match for the sprained-foot query:
- Best match for the vomiting query:
- What happened with "someone collapsed and is not breathing", and did the CPR FAQ appear in the top 3?

NumPy was used for the answer-length statistics, the Boolean mask that flags weak matches, and the per-query summary. Without cleaning, the repeated question would have been stored twice, and the missing answer would have caused an error.

## Key takeaways
- **Python functions** keep loading, cleaning, chunking and searching separate, so each can be tested and reused.
- **Pandas** holds the table; **NumPy** does the measuring and flagging.
- **Embeddings** turn text into numbers, and **Chroma** finds the closest ones to your question.
- **Chunking** trades off context against noise: too big drowns the answer, too small loses context.

**Try this:**
1. Change `CHUNK_SIZE` to `200`, rerun from section 7, and explain why the chunk count drops to about 48.
2. Search for `"how do I do CPR"` and compare it with `"someone collapsed and is not breathing"`. Why does the CPR FAQ rank differently?
3. Change `WEAK_MATCH_DISTANCE` from `1.0` to `0.8` and see how many matches get flagged.
4. Change `TOP_K` to `5` and look at the `distinct_faqs` column.